# 02 - Skill Matcher (hybrid)
**Task:** the user types skills in free text ("py", "pyhton", "making charts from data").
We must map each one to a standard skill from `skills.csv`, or say "no match".

Three layers, tried in order (code in `skill_matcher.py`):
1. **Alias lookup**: normalized text matches a skill name or alias exactly
2. **Fuzzy match**: typos ("pyhton" -> Python), edit-distance based
3. **Embedding similarity**: meaning-based ("cleaning messy datasets" -> Data Cleaning)

Embedding matches with medium similarity are returned with `needs_confirmation = True`
so the website asks the user, instead of silently guessing.

In [ ]:
import pandas as pd
from skill_matcher import SkillMatcher, normalize, split_skills

skills = pd.read_csv("../data/skills.csv")
test = pd.read_csv("../data/skill_match_test.csv").fillna("")
print(len(skills), "skills |", len(test), "test inputs")
test["kind"].value_counts()

## Test set
`skill_match_test.csv` has 4 kinds of input:
- **alias**: known short forms ("js", "sklearn")
- **typo**: spelling mistakes ("javscript")
- **phrase**: descriptions, not skill names ("writing unit tests")
- **none**: skills we do NOT have ("photoshop", "java"); the correct answer is *no match*

In [ ]:
def outcome(row):
    exp = row["expected"]
    got = "" if pd.isna(row["skill_id"]) else row["skill_id"]
    if exp == "":
        if row["method"] == "none":
            return "correct_none"
        return "false_ask" if row["needs_confirmation"] else "false_match"
    if got == exp:
        return "correct_asked" if row["needs_confirmation"] else "correct_auto"
    if got == "":
        return "missed"
    return "wrong_asked" if row["needs_confirmation"] else "wrong_auto"

def evaluate(matcher):
    res = pd.DataFrame(matcher.match_many(test["raw_text"]))
    res["expected"] = test["expected_skill_id"].values
    res["kind"] = test["kind"].values
    res["outcome"] = res.apply(outcome, axis=1)
    return res

def summary(res):
    c = res["outcome"].value_counts()
    cols = ["correct_auto", "correct_asked", "correct_none", "missed", "wrong_asked", "wrong_auto", "false_ask", "false_match"]
    out = {k: int(c.get(k, 0)) for k in cols}
    out["good_%"] = round(100 * (out["correct_auto"] + out["correct_asked"] + out["correct_none"]) / len(res), 1)
    out["bad_auto"] = out["wrong_auto"] + out["false_match"]   # silent mistakes: the worst kind
    return out

## 1. Ablation: what does each layer add?
- **correct_auto**: right answer, accepted automatically
- **correct_asked**: right answer, but the user is asked to confirm
- **correct_none**: correctly said "no match"
- **missed**: should have matched but returned nothing
- **bad_auto**: silent mistakes (wrong skill accepted, or a fake match) - this must stay near 0

In [ ]:
A = evaluate(SkillMatcher(skills, fuzzy_threshold=None))
B = evaluate(SkillMatcher(skills))
pd.DataFrame({"A: alias only": summary(A), "B: alias + fuzzy": summary(B)}).T

In [ ]:
# Which kinds of input does each setup handle?
pd.DataFrame({
    "A": A.groupby("kind")["outcome"].apply(lambda s: s.isin(["correct_auto", "correct_none"]).mean()),
    "B": B.groupby("kind")["outcome"].apply(lambda s: s.isin(["correct_auto", "correct_none"]).mean()),
}).round(2)

## 2. Fuzzy threshold
Lower = catches more typos but risks false matches. We want the lowest value with `bad_auto = 0`.

In [ ]:
rows = []
for th in [95, 90, 85, 80, 75, 70]:
    s = summary(evaluate(SkillMatcher(skills, fuzzy_threshold=th)))
    rows.append({"fuzzy_threshold": th, **s})
pd.DataFrame(rows)

## 3. Embedding layer
Needs `pip install sentence-transformers`. The first run downloads a small model (~90 MB).
If it fails, the notebook keeps working with layers 1-2 only.

In [ ]:
try:
    from skill_matcher import load_sentence_encoder
    encoder = load_sentence_encoder("all-MiniLM-L6-v2")
    print("embedding model loaded")
except Exception as e:
    encoder = None
    print("Embedding model not available, skipping layer 3:", type(e).__name__, e)

In [ ]:
if encoder is not None:
    mC = SkillMatcher(skills, encoder=encoder)
    C = evaluate(mC)
    display(pd.DataFrame({"A: alias only": summary(A), "B: alias + fuzzy": summary(B),
                          "C: alias + fuzzy + embedding": summary(C)}).T)

## 4. Embedding thresholds
- similarity >= `emb_high`: accepted automatically
- `emb_low` <= similarity < `emb_high`: the user is asked to confirm
- below `emb_low`: no match

Pick the setting where `bad_auto` is lowest while `good_%` stays high.

In [ ]:
if encoder is not None:
    rows = []
    for high in [0.5, 0.6, 0.7, 0.8]:
        for low in [0.3, 0.4, 0.5]:
            if low >= high:
                continue
            mC.emb_high, mC.emb_low = high, low
            rows.append({"emb_high": high, "emb_low": low, **summary(evaluate(mC))})
    display(pd.DataFrame(rows))

## 5. Where does it still fail?

In [ ]:
final = C if encoder is not None else B
final[final["outcome"].isin(["missed", "wrong_auto", "wrong_asked", "false_match", "false_ask"])][
    ["raw_text", "expected", "skill_id", "method", "score", "outcome"]]

## 6. Try your own inputs
Users often type several skills in one box, so the backend first splits the text with `split_skills`.

In [ ]:
print(split_skills("python, html and css & sql"))

In [ ]:
matcher = mC if encoder is not None else SkillMatcher(skills)
for t in [x for field in ["py, Pyhton", "html and css", "machine learnin", "making apps talk to servers", "photoshop, excel"] for x in split_skills(field)]:
    r = matcher.match(t)
    print(f"{t!r:35} -> {r['skill_name']} | {r['method']} | score={r['score']} | confirm={r['needs_confirmation']}")